# Imports

In [ ]:
import sqlite3
import os
import datetime

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# Setup the DB

In [ ]:
DB_PATH = "tasks.db"

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

## Two tables

- independent, for simplicity
- one for storing the tasks
- another for storing the steps

In [ ]:
cursor.execute("""
CREATE TABLE tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    description TEXT
);
""")

cursor.execute("""
CREATE TABLE task_steps (
    task_id INTEGER,
    step_id INTEGER,
    step_description TEXT
);
""")

conn.commit()

## Helper to View DB

In [ ]:
def show_all_records():
    cursor.execute("SELECT id, description FROM tasks ORDER BY id")
    tasks = cursor.fetchall()

    cursor.execute("SELECT task_id, step_id, step_description FROM task_steps ORDER BY task_id, step_id")
    steps = cursor.fetchall()

    step_map = {}
    for task_id, step_id, step_description in steps:
        if task_id not in step_map:
            step_map[task_id] = []
        step_map[task_id].append((step_id, step_description))

    list_output = []
    for task_id, task_desc in tasks:
        list_output.append(f"Task {task_id}: {task_desc}")

        if task_id in step_map:
            for step_id, step_desc in step_map[task_id]:
                list_output.append(f"\tStep {step_id}: {step_desc}")

        else:
            list_output.append(f"\t(No steps added yet)")

        list_output.append("")

    print("\n".join(list_output))

In [ ]:
show_all_records()

# nothing right now!

# Setup the Agent

## LLM

- system message sets the style / persona of the agent

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    messages=[
        {
            "role": "system",
            "content": "You are an AI that plans tasks in detailed steps. Each step must contain 3–5 sentences explaining the action thoroughly. Always use tools to create tasks, add steps, and log reasoning. You must use the log tool frequently to document your internal reasoning. Each log entry should contain multi-sentence reflections about your choices, planning considerations, and reasoning steps that help the developer understand how you arrived at each task step."
        }
    ]
)

/usr/local/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3473: UserWarning: WARNING! messages is not default parameter.
                messages was transferred to model_kwargs.
                Please confirm that messages is what you intended.
  if (await self.run_code(code, result,  async_=asy)):


## Tools

- one for adding the task to the db
- another for adding a steap for a task to the db
- a third one to log the LLM's reasoning in a log file

Note that we are creating the tools using the `@tool` decorator, which is a more convenient method   
compared to the previous method we have seen where the function is defined and then `Tool()` is used separately.  
Under the hood though, they work similarly.  
  
Also, in this method, the docstring is crucial for the LLM to understand the tool.

In [ ]:
@tool
def create_task(description: str) -> str:
    """
    Create a new task in the SQLite database.
    Input:
        description (str): The main task description.
    Returns:
        str: NEW_TASK_ID:<id>
    """
    cursor.execute("INSERT INTO tasks (description) VALUES (?)", (description,))
    conn.commit()
    task_id = cursor.lastrowid
    return f"NEW_TASK_ID:{task_id}"

In [ ]:
@tool
def add_step_task(task_id: int, step_text: str) -> str:
    """
    Add a detailed planning step to a task.
    Inputs:
        task_id (int): ID of the task.
        step_text (str): The detailed 3–5 sentence step description.
    Returns:
        str: Confirmation message.
    """
    cursor.execute("SELECT MAX(step_id) FROM task_steps WHERE task_id=?", (task_id,))
    row = cursor.fetchone()
    next_step_id = 1 if row[0] is None else row[0] + 1

    cursor.execute(
        "INSERT INTO task_steps (task_id, step_id, step_description) VALUES (?, ?, ?)",
        (task_id, next_step_id, step_text)
    )
    conn.commit()
    return f"Step {next_step_id} added to task {task_id}: {step_text}"


In [ ]:
@tool
def log(msg: str) -> str:
    """
    Record detailed internal reasoning steps, including:
    - why the next step was chosen,
    - what sub-decisions were evaluated,
    - details about constraints, alternatives, or assumptions,
    - planning considerations that do NOT go into the final user-facing step description.

    Use this tool often during planning so the log file contains a rich trace of your thought process.

    Input:
        msg (str): A multi-sentence explanation of behind-the-scenes reasoning.
    Returns:
        str: "logged"
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("agent_trace.log", "a") as f:
        f.write(f"[{timestamp}] {msg}\n")
    print(f"\n### TRACE: {msg}")
    return "logged"

## Initialize

- we are using `OPENAI_FUNCTIONS` AgentType, instead of `ZERO_SHOT_REACT_DESCRIPTION`
> The REACT type is designed for short responses only, so our requirement of more descriptive steps wont get fulfilled
- setting `max_iterations=50`
> because there are 3 tools, the number of back-forth interactions between the LLM and tools can exceed the default number of iterations (usually 20)

In [ ]:
agent = initialize_agent(
    tools=[create_task, add_step_task, log],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
    max_iterations=50
)

/tmp/ipykernel_91/2613452026.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


## Task Prompt

In [ ]:
prompt = """
You are an AI assistant that creates detailed, multi-step plans for tasks and goals.

You have access to three tools:

1. create_task(description: str)
   - Creates a new task in the database.
   - Returns NEW_TASK_ID:<id>.

2. add_step_task(task_id: int, step_text: str)
   - Adds a detailed step (3–5 sentences) to the task.
   - Each step must be clear, descriptive, and explain the reasoning.

3. log(msg: str)
   - Logs your internal reasoning, planning considerations, and decision-making.
   - This tool is ONLY for internal notes and will not be shown to end-users.
   - Your log messages should be 2–4 sentences.
   - Use this tool OFTEN to provide a rich trace of reasoning.

---------------------------------------------
PLANNING INSTRUCTIONS
---------------------------------------------

For the task or goal provided:

1. First call create_task(description) to store the main task in the database.

2. Then produce a detailed step-by-step plan.
   - Each step must be 3–5 sentences long.
   - Each step must explain WHY it is important, HOW it will be done,
     and WHAT considerations matter for timing, resources, people, or constraints.
   - After writing each step, use add_step_task(task_id, step_text).

3. Before adding ANY step, ALWAYS call log(msg) with:
   - Why this step is needed,
   - What alternatives you considered,
   - Why this step should come next,
   - Any assumptions or constraints you are using.
   These log messages provide reasoning and must be 2–4 sentences each.

4. You must repeat this cycle until you have created a complete, thoughtful plan:
   - log() reasoning
   - add_step_task() for the detailed step

5. When the plan is complete, stop.

---------------------------------------------
TASK TO PLAN:
{question}
---------------------------------------------
"""

# Use the Agent

In [ ]:
user_task = "Plan a week-long trip to Japan focusing on cultural experiences."

result = agent.run(prompt.replace("{question}", user_task))

print("\n===RESPONSE FROM AGENT===")
print(result)

/tmp/ipykernel_91/1957222545.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = agent.run(prompt.replace("{question}", user_task))


> Entering new AgentExecutor chain...

Invoking: `create_task` with `{'description': 'Plan a week-long trip to Japan focusing on cultural experiences.'}`


NEW_TASK_ID:1
Invoking: `log` with `{'msg': "The first step in planning the trip to Japan is to define the main cultural experiences we want to include. I considered starting with a general itinerary, but it's crucial to first understand what specific cultural activities resonate with the travelers. This step is necessary to ensure that the overall plan aligns with the travelers' interests and to create a focused itinerary. I assume that the travelers are interested in traditional experiences such as tea ceremonies, temple visits, and local festivals."}`



### TRACE: The first step in planning 

In [ ]:
show_all_records()

Task 1: Plan a week-long trip to Japan focusing on cultural experiences.
	Step 1: Identify and list specific cultural experiences that travelers want to include in the itinerary. This could involve activities such as attending a traditional tea ceremony, visiting historical temples, exploring local markets, and participating in festivals. By understanding the travelers' interests, we can tailor the trip to ensure it is meaningful and engaging. This step is important because it sets the foundation for the entire trip planning process.
	Step 2: Research and compile a list of the best locations and events for the identified cultural experiences. This involves looking into specific cities in Japan that are known for their cultural heritage, such as Kyoto for temples and tea ceremonies, and Tokyo for festivals and markets. Additionally, checking the timing of events to ensure they align with the trip dates is essential. This step is necessary to create a realistic itinerary that maximizes c

In [ ]:
agent = initialize_agent(
    tools=[create_task, add_step_task, log],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=False,
    max_iterations=50
)

In [ ]:
user_task = "Organize a study schedule for my Quantum Computing final exams next month."

result = agent.run(prompt.replace("{question}", user_task))

print("\n===RESPONSE FROM AGENT===")
print(result)


### TRACE: The first step in organizing a study schedule for the Quantum Computing final exams is to assess the syllabus and topics that need to be covered. This is crucial to ensure that the study plan is comprehensive and focused on the relevant material. I considered starting with a review of personal strengths and weaknesses in the subject, but it's more effective to first outline the entire syllabus. This will guide the scheduling process and help identify the time needed for each topic.

### TRACE: The next step is to evaluate how much time is available until the exam date. This is essential to create a realistic study schedule that fits within the available timeframe. I considered using a calendar for this, but a simple calculation of days left and a rough estimate of hours per day will suffice. This will help in determining how many study sessions can be scheduled and how to distribute the topics over time.

### TRACE: The next step is to prioritize the topics based on their d

In [ ]:
show_all_records()

Task 1: Plan a week-long trip to Japan focusing on cultural experiences.
	Step 1: Identify and list specific cultural experiences that travelers want to include in the itinerary. This could involve activities such as attending a traditional tea ceremony, visiting historical temples, exploring local markets, and participating in festivals. By understanding the travelers' interests, we can tailor the trip to ensure it is meaningful and engaging. This step is important because it sets the foundation for the entire trip planning process.
	Step 2: Research and compile a list of the best locations and events for the identified cultural experiences. This involves looking into specific cities in Japan that are known for their cultural heritage, such as Kyoto for temples and tea ceremonies, and Tokyo for festivals and markets. Additionally, checking the timing of events to ensure they align with the trip dates is essential. This step is necessary to create a realistic itinerary that maximizes c